In [2]:
# ============================================
# DATA ANALYST AGENT WITH CODE EXECUTION
# ============================================
# This notebook demonstrates how to create an AI agent that can:
# - Execute Python code dynamically
# - Perform data analysis tasks
# - Handle data cleaning operations

# Import required libraries
from crewai import Agent, Task, Crew  # Core CrewAI components
from crewai_tools import CodeInterpreterTool  # Tool for executing Python code

import os

# Set up API key - Replace with your actual OpenAI API key
# os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
# ================================================================================
# Configure LLM
# ================================================================================

# Set up API key (uncomment and replace with your key if not using .env)
# os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"

# Initialize GPT-4 for advanced reasoning
# llm = LLM(model="gpt-4")
from helpers import get_llm

llm = get_llm(model = "groq/qwen/qwen3-32b")

LLM initialized: groq/qwen/qwen3-32b (via databricks)


In [19]:
# ============================================
# AGENT DEFINITION
# ============================================
# Create a data analyst agent with code execution capabilities

data_analyst_agent = Agent(
    llm = llm,
    role="General Data Analyst",  # Define the agent's role
    goal="Perform data analysis tasks based on user inputs and instructions and execute them using Python",
    backstory="You are an experienced data analyst skilled in Python and data manipulation",
    
    # KEY FEATURE: Allow this agent to execute Python code
    # This enables the agent to perform actual data operations
    # allow_code_execution=True,
    
    # Set retry limit in case of errors
    max_retry_limit=3,
    
    # IMPORTANT: Provide CodeInterpreterTool for Python execution
    # This tool allows the agent to run pandas, numpy, and other data libraries
    tools=[CodeInterpreterTool()]
)

In [20]:
# ============================================
# TASK FACTORY FUNCTION
# ============================================
# This function creates dynamic tasks based on user requirements
# This pattern is useful when you need to create multiple similar tasks
# with different parameters

def create_data_analysis_task(dataframe_path: str, user_instruction: str, output_path: str):
    """
    Creates a data analysis task with specific parameters.
    
    Parameters:
    - dataframe_path: Path to input CSV file
    - user_instruction: Natural language instruction for data operation
    - output_path: Path where the result should be saved
    
    Returns: Task object configured for the data analyst agent
    """
    return Task(
        # Description: Clear instructions for what the agent should do
        description=(
            f"Load the dataframe from '{dataframe_path}', perform the following task: '{user_instruction}',"
            f"and save the resulting dataframe to '{output_path}'. Execute the task using Python, if you are"
            f"not able to execute the code on the machine please explain why."
        ),
        
        # Expected output: What success looks like
        expected_output=f"Dataframe with the applied operations saved to '{output_path}'.",
        
        # Assign this task to the data analyst agent
        agent=data_analyst_agent,
    )

In [21]:
# ============================================
# CONFIGURATION: Define Task Parameters
# ============================================
# Set up the specific task you want the agent to perform

# Path to your input CSV file (should be in the same directory or provide full path)
dataframe_path = "./docs/input_data.csv"

# Natural language instruction - The agent will interpret and execute this
# Examples: "fix missing values", "remove duplicates", "normalize column X"
user_instruction = "fix missing values"

# Where to save the processed data
output_path = "output_data.csv"

In [22]:
# ============================================
# CREATE THE TASK
# ============================================
# Use the factory function to create a task with our parameters

data_analysis_task = create_data_analysis_task(dataframe_path, user_instruction, output_path)

In [23]:
# ============================================
# CREW SETUP
# ============================================
# Create a Crew to orchestrate the agent and task
# A Crew manages the execution flow and coordinates between agents

analyst_crew = Crew(
    agents=[data_analyst_agent],  # List of agents (can have multiple)
    tasks=[data_analysis_task]     # List of tasks to execute
)

In [24]:
# ============================================
# EXECUTE THE CREW
# ============================================
# Start the crew execution - the agent will:
# 1. Read the description and understand the task
# 2. Generate Python code to solve the problem
# 3. Execute the code using CodeInterpreterTool
# 4. Save the results and provide output

# NOTE: Variable name should be 'analyst_crew' not 'analysis_crew'
result = analyst_crew.kickoff()

# Display the final result
print(result)

# WHAT HAPPENED:
# - The agent loaded the CSV file
# - Detected missing values in numeric and categorical columns
# - Applied appropriate filling strategies (mean for numeric, mode for categorical)
# - Saved the cleaned dataframe to output_data.csv

 Docker is not installed
 Running code in restricted sandbox
    ID       Name             Age          Values Category
0    1   Person_1 -1000000.000000    86313.621176        D
1    2   Person_2       34.000000       92.509060        C
2    3   Person_1       63.000000       40.235452        A
3    4   Person_4  1000000.000000   186355.976883        C
4    5   Person_5       63.000000  1000000.000000        B
5    6   Person_1       40.000000  1000000.000000        C
6    7   Person_7       36.000000       85.596729        C
7    8   Person_8       61.000000    15848.043479        C
8    9   Person_1       32.000000       22.883002        C
9   10  Person_10    74127.975324       90.895506        D
10  11  Person_11       49.000000   186355.976883        B
11  12  Person_12       53.000000       84.734297        A
12  13  Person_13       62.000000       17.211174        D
13  14  Person_14       26.000000  1000000.000000        C
14  15   Person_1    46721.568341       74.471218     

<string>:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


<string>:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform 

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: 51b01b73-1637-4b83-b21b-503f828a8ebb                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/51b01b73-1637-4b83-b21b-503f828a8ebb?access_code=TRA │
│ CE-60570fc944                                                                                                   │
│ 🔑 Access Code: TRACE-60570fc944                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The code to fix missing values and save the dataframe to 'output_data.csv' was executed. However, the Code Interpreter environment does not provide access to the file system to retrieve the saved CSV content directly. The resulting dataframe is stored in 'output_data.csv' as requested.
